# Prepare the RL-ready dataset from the UC competition files

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import gzip
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

DATA_ROOT = Path(__import__("os").environ.get("UC_RAW_DATA", "../starting_kit_ai-uc_v2_warm-up/Train_Data"))

OUT_ROOT = Path(__import__("os").environ.get("UC_DATA_ROOT", "../rl_ready_competition_data"))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

HORIZON = 72

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
RNG_SEED = 42

print("DATA_ROOT:", DATA_ROOT)
print("OUT_ROOT:", OUT_ROOT)

## 2. Scan the instance folders

In [ ]:
instance_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("instance_")])
print(f"Found {len(instance_dirs)} instance folders.")
print("First five folders:")
for p in instance_dirs[:5]:
    print(" -", p.name)

if not instance_dirs:
    raise FileNotFoundError(f"No instance folders found in {DATA_ROOT}")

## 3. Helpers to read one instance folder

In [ ]:
def read_excel_any(path: Path) -> dict[str, pd.DataFrame]:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_excel(path, sheet_name=None)


def read_gz_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(path)
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def find_file(folder: Path, stem: str):
    candidates = []
    for p in folder.iterdir():
        if p.is_file() and p.name.lower().startswith(stem.lower()):
            candidates.append(p)
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda p: (len(p.name), p.suffix.lower()))
    return candidates[0]


def standardize_sheet_name(name: str) -> str:
    return str(name).strip().lower().replace(" ", "_")


def read_instance(folder: Path) -> dict:
    exp_path = find_file(folder, "explanatory_variables")
    resp_path = find_file(folder, "response_variables")
    inp_path = find_file(folder, "InputData")
    out_path = find_file(folder, "OutputData")

    if exp_path is None:
        raise FileNotFoundError(f"Missing explanatory_variables.xlsx in {folder}")
    if resp_path is None:
        raise FileNotFoundError(f"Missing response_variables.xlsx in {folder}")
    if inp_path is None:
        raise FileNotFoundError(f"Missing InputData.json.gz in {folder}")
    if out_path is None:
        raise FileNotFoundError(f"Missing OutputData.json.gz in {folder}")

    explanatory = read_excel_any(exp_path)
    response = read_excel_any(resp_path)
    input_json = read_gz_json(inp_path)
    output_json = read_gz_json(out_path)

    return {
        "instance_id": folder.name,
        "folder": str(folder),
        "explanatory": explanatory,
        "response": response,
        "input_json": input_json,
        "output_json": output_json,
    }


sample = read_instance(instance_dirs[0])
print("Loaded example instance:", sample["instance_id"])
print("Explanatory sheets:", list(sample["explanatory"].keys()))
print("Response sheets:", list(sample["response"].keys()))
print("Input JSON keys:", list(sample["input_json"].keys())[:20])
print("Output JSON keys:", list(sample["output_json"].keys())[:20])

In [ ]:
def input_json_to_dfs(input_json):
    rows = []

    params = input_json.get("Parameters", {})
    param_df = pd.DataFrame([{"section": "Parameters", "key": k, "value": v} for k, v in params.items()])

    gens = input_json.get("Generators", {})
    gen_rows = []
    for name, info in gens.items():
        if isinstance(info, dict):
            row = {"section": "Generators", "name": name}
            row.update(info)
            gen_rows.append(row)
    gen_df = pd.DataFrame(gen_rows)

    stor = input_json.get("Storage units", {})
    stor_rows = []
    for name, info in stor.items():
        if isinstance(info, dict):
            row = {"section": "Storage units", "name": name}
            row.update(info)
            stor_rows.append(row)
    stor_df = pd.DataFrame(stor_rows)

    buses = input_json.get("Buses", {})
    bus_rows = []
    for name, info in buses.items():
        if isinstance(info, dict):
            row = {"section": "Buses", "name": name}
            row.update(info)
            bus_rows.append(row)
    bus_df = pd.DataFrame(bus_rows)

    return {
        "parameters_df": param_df,
        "generators_df": gen_df,
        "storage_df": stor_df,
        "buses_df": bus_df,
    }

def output_json_to_dfs(output_json):
    out_rows = []

    for key, value in output_json.items():
        if isinstance(value, list):
            tmp = pd.DataFrame({
                "hour": np.arange(1, len(value) + 1),
                key: value
            })
            out_rows.append((key, tmp))
        elif isinstance(value, dict):
            df = pd.DataFrame(value)
            df.insert(0, "key", key)
            out_rows.append((key, df))
        else:
            out_rows.append((key, pd.DataFrame([{"key": key, "value": value}])))
    
    return out_rows

In [ ]:
json_dfs = input_json_to_dfs(sample["input_json"])

display(json_dfs["parameters_df"])
display(json_dfs["generators_df"].head())
display(json_dfs["storage_df"].head())
display(json_dfs["buses_df"].head())

In [ ]:
output_tables = output_json_to_dfs(sample["output_json"])

for name, df in output_tables[:5]:
    print("Table:", name)
    display(df.head())

## Data VIZ

In [ ]:
sample = read_instance(instance_dirs[0])
gens = sample["input_json"].get("Generators", {})
for name, info in gens.items():
    if isinstance(info, dict) and str(info.get("Type", "")).strip().lower() == "thermal":
        print(f"Generator: {name}")
        for k, v in info.items():
            val_preview = str(v)[:100] if not isinstance(v, (int, float)) else v
            print(f"  {k}: {val_preview}")
        break

In [ ]:
sample = read_instance(instance_dirs[0])
gens = sample["input_json"].get("Generators", {})

gen_rows = []
for name, info in gens.items():
    if not isinstance(info, dict):
        continue
    if str(info.get("Type", "")).strip().lower() != "thermal":
        continue

    pcurve_mw = info.get("Production cost curve (MW)", [])
    pcurve_usd = info.get("Production cost curve ($)", [])
    pmax = float(max(pcurve_mw)) if pcurve_mw else 0.0
    pmin = float(pcurve_mw[0]) if pcurve_mw else 0.0

    su_list = info.get("Startup costs ($)", [])
    startup = float(su_list[0]) if su_list else 0.0

    ramp_up = float(info.get("Ramp up limit (MW)", pmax))
    ramp_down = float(info.get("Ramp down limit (MW)", pmax))
    min_up = float(info.get("Minimum uptime (h)", 1))
    min_down = float(info.get("Minimum downtime (h)", 1))

    if len(pcurve_mw) >= 2 and len(pcurve_usd) >= 2:
        dmw = float(pcurve_mw[-1]) - float(pcurve_mw[-2])
        dusd = float(pcurve_usd[-1]) - float(pcurve_usd[-2])
        marginal = dusd / dmw if dmw > 1e-6 else 0.0
    else:
        marginal = 0.0

    gen_rows.append({
        "generator": name,
        "pmin": pmin, "pmax": pmax,
        "startup_cost": startup,
        "marginal_cost": marginal,
        "ramp_up": ramp_up, "ramp_down": ramp_down,
        "min_up": min_up, "min_down": min_down,
    })

gen_df = pd.DataFrame(gen_rows)
gen_df["short_name"] = gen_df["generator"].str[:12]
gen_df = gen_df.sort_values("pmax", ascending=False).reset_index(drop=True)

print(f"Generator Fleet: {len(gen_df)} thermal units")
print(f"Total Pmax: {gen_df['pmax'].sum():,.0f} MW")
print(f"Total Pmin: {gen_df['pmin'].sum():,.0f} MW")
print(f"Pmax range: {gen_df['pmax'].min():.0f} – {gen_df['pmax'].max():.0f} MW")

fig, ax = plt.subplots(figsize=(18, 8))
x = np.arange(len(gen_df))
w = 0.4

ax.bar(x - w/2, gen_df["pmax"], width=w, color="#2D5F8A", alpha=0.85, label="Pmax")
ax.bar(x + w/2, gen_df["pmin"], width=w, color="#E07B39", alpha=0.85, label="Pmin (must-run)")

ax.set_xticks(x)
ax.set_xticklabels(gen_df["short_name"], rotation=90, fontsize=7)
ax.set_ylabel("Capacity (MW)")
ax.set_title("Generator Fleet — Pmin and Pmax (sorted by Pmax)", fontsize=13)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.0), fontsize=10,
          framealpha=0.9)
ax.grid(True, alpha=0.2, axis="y")

ax.text(0.98, 0.95, f"Total Pmax: {gen_df['pmax'].sum():,.0f} MW\n"
        f"Total Pmin: {gen_df['pmin'].sum():,.0f} MW\n"
        f"Fleet size: {len(gen_df)} units",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7))
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(gen_df["pmax"], gen_df["startup_cost"],
                     s=gen_df["pmin"] * 2 + 20,
                     c=gen_df["marginal_cost"], cmap="RdYlGn_r",
                     alpha=0.8, edgecolors="black", linewidth=0.5)
cb = fig.colorbar(scatter, ax=ax, shrink=0.8)
cb.set_label("Marginal Cost ($/MWh)")

top5 = gen_df.nlargest(5, "startup_cost")
for _, row in top5.iterrows():
    ax.annotate(row["short_name"], (row["pmax"], row["startup_cost"]),
                fontsize=8, ha="left", va="bottom",
                xytext=(5, 5), textcoords="offset points")

ax.set_xlabel("Pmax (MW)")
ax.set_ylabel("Startup Cost ($)")
ax.set_title("Startup Cost vs Capacity (bubble size = Pmin)", fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Operational Constraints Across the 51-Generator Fleet", fontsize=13)

ax = axes[0]
bins = np.arange(0.5, gen_df["min_up"].max() + 1.5, 1)
ax.hist(gen_df["min_up"], bins=bins, color="purple", alpha=0.7,
        label="Min Uptime", edgecolor="white")
ax.hist(gen_df["min_down"], bins=bins, color="pink", alpha=0.7,
        label="Min Downtime", edgecolor="white")
ax.set_xlabel("Hours")
ax.set_ylabel("Number of Generators")
ax.set_title("Min Uptime / Downtime Distribution")
ax.legend()
ax.grid(True, alpha=0.2, axis="y")

ax = axes[1]
ax.scatter(gen_df["pmax"], gen_df["ramp_up"], s=40, color="#4AA96C",
           alpha=0.7, label="Ramp Up")
ax.scatter(gen_df["pmax"], gen_df["ramp_down"], s=40, color="#C75B7A",
           alpha=0.7, marker="v", label="Ramp Down")
ax.plot([0, gen_df["pmax"].max()], [0, gen_df["pmax"].max()],
        "k--", alpha=0.3, label="1:1 line")
ax.set_xlabel("Pmax (MW)")
ax.set_ylabel("Ramp Rate (MW/h)")
ax.set_title("Ramp Rates vs Capacity")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

ax = axes[2]
ratio = gen_df["pmin"] / gen_df["pmax"].clip(lower=1e-6) * 100
ax.hist(ratio, bins=20, color="#6C5B9E", alpha=0.7, edgecolor="white")
ax.axvline(ratio.median(), color="red", ls="--", lw=2,
           label=f"Median: {ratio.median():.0f}%")
ax.set_xlabel("Pmin / Pmax (%)")
ax.set_ylabel("Number of Generators")
ax.set_title("Must-Run Ratio (Pmin/Pmax)")
ax.legend()
ax.grid(True, alpha=0.2, axis="y")

plt.tight_layout()
plt.show()

gen_df["category"] = pd.cut(gen_df["pmax"],
                             bins=[0, 100, 500, 1000, float("inf")],
                             labels=["Small (<100MW)", "Medium (100-500MW)",
                                     "Large (500-1000MW)", "Very Large (>1000MW)"])

cat_summary = gen_df.groupby("category", observed=True).agg(
    count=("generator", "count"),
    total_pmax=("pmax", "sum"),
    total_pmin=("pmin", "sum"),
    avg_startup=("startup_cost", "mean"),
    avg_min_up=("min_up", "mean"),
    avg_marginal=("marginal_cost", "mean"),
).round(1)
cat_summary["pmax_share"] = (cat_summary["total_pmax"] /
                              cat_summary["total_pmax"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(14, 3))
ax.axis("off")
table_data = []
for cat in cat_summary.index:
    r = cat_summary.loc[cat]
    table_data.append([
        cat, f"{r['count']:.0f}", f"{r['total_pmax']:,.0f}",
        f"{r['pmax_share']:.1f}%", f"{r['total_pmin']:,.0f}",
        f"${r['avg_startup']:,.0f}", f"{r['avg_min_up']:.1f}h",
        f"${r['avg_marginal']:.1f}"
    ])
col_labels = ["Category", "Count", "Total Pmax (MW)", "% of Fleet",
               "Total Pmin (MW)", "Avg Startup ($)", "Avg Min Up",
               "Avg Marg. Cost"]
tbl = ax.table(cellText=table_data, colLabels=col_labels, loc="center",
               cellLoc="center", colColours=["#2D5F8A"]*len(col_labels))
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8)
for key, cell in tbl.get_celld().items():
    if key[0] == 0:
        cell.set_text_props(color="white", fontweight="bold")
    cell.set_edgecolor("#D0D0D0")
ax.set_title("Generator Fleet Summary by Size Category", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

## 4. Inspect one instance in a human-friendly way

In [ ]:
def first_sheet(df_dict: dict[str, pd.DataFrame], keywords=()):
    for key, df in df_dict.items():
        low = key.lower()
        if all(k.lower() in low for k in keywords):
            return key, df
    key = next(iter(df_dict))
    return key, df_dict[key]


exp_key, exp_df = first_sheet(sample["explanatory"], keywords=("profile",))
init_key, init_df = first_sheet(sample["explanatory"], keywords=("initial",))
resp_key, resp_df = first_sheet(sample["response"])

exp_df.rename(columns={"Unnamed: 0": "hour"}, inplace=True)
init_df.rename(columns={"Unnamed: 0": "generator"}, inplace=True)
resp_df.rename(columns={"Unnamed: 0": "hour"}, inplace=True)

display(pd.DataFrame({
    "file": ["explanatory profile sheet", "explanatory initial conditions", "response variables"],
    "sheet_name": [exp_key, init_key, resp_key],
    "shape": [exp_df.shape, init_df.shape, resp_df.shape],
}))

print("Profile sheet preview:")
display(exp_df.head())
print("Initial conditions preview:")
display(init_df.head())
print("Response variables preview:")
display(resp_df.head())

## 4A. Quick EDA overview for presentation

In [ ]:
import re


def parse_instance_name(name: str):
    m = re.match(r"instance_(\d{4})_Q(\d)_(\d+)", name)
    if not m:
        return None
    return {
        "instance_id": name,
        "year": int(m.group(1)),
        "quarter": int(m.group(2)),
        "index": int(m.group(3)),
    }


meta_rows = [parse_instance_name(p.name) for p in instance_dirs]
meta_rows = [x for x in meta_rows if x is not None]
meta_df = pd.DataFrame(meta_rows)
display(meta_df.head())

print(f"Total instances: {len(meta_df)}")
print("Instances per year:")
display(meta_df.groupby("year").size().rename("count").to_frame())
print("Instances per quarter:")
display(meta_df.groupby(["year", "quarter"]).size().rename("count").reset_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
meta_df.groupby("year").size().plot(kind="bar", ax=axes[0], color="tab:blue")
axes[0].set_title("Number of instances by year")
axes[0].set_xlabel("year")
axes[0].set_ylabel("count")

quarter_counts = meta_df.groupby("quarter").size().sort_index()
quarter_counts.plot(kind="bar", ax=axes[1], color="tab:orange")
axes[1].set_title("Number of instances by quarter")
axes[1].set_xlabel("quarter")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

year_quarter_table = (
    meta_df.groupby(["year", "quarter"])
    .size()
    .reset_index(name="count")
    .pivot(index="year", columns="quarter", values="count")
    .fillna(0)
    .astype(int)
)

display(year_quarter_table)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    year_quarter_table,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    linewidths=0.5,
    cbar_kws={"label": "number of instances"},
    ax=ax
)
ax.set_title("Number of Instances per Year and Quarter")
ax.set_xlabel("Quarter")
ax.set_ylabel("Year")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
year_quarter_table.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    colormap="tab20"
)
ax.set_title("Instances per Year and Quarter")
ax.set_xlabel("Year")
ax.set_ylabel("Number of instances")
ax.legend(title="Quarter", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
exp_plot = exp_df.copy()
init_plot = init_df.copy()
resp_plot = resp_df.copy()

if "Unnamed: 0" in exp_plot.columns:
    exp_plot = exp_plot.rename(columns={"Unnamed: 0": "hour"})
if "Unnamed: 0" in init_plot.columns:
    init_plot = init_plot.rename(columns={"Unnamed: 0": "generator"})
if "Unnamed: 0" in resp_plot.columns:
    resp_plot = resp_plot.rename(columns={"Unnamed: 0": "hour"})

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(exp_plot["demand"], label="demand", lw=2)
ax.plot(exp_plot["wind"], label="wind", lw=2)
ax.plot(exp_plot["solar"], label="solar", lw=2)
ax.plot(exp_plot["wind"] + exp_plot["solar"], label="wind + solar", lw=2, linestyle="--")
ax.set_title(f"One 72-hour profile example: {sample['instance_id']}")
ax.set_xlabel("hour")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

if "initial_status" in init_plot.columns:
    status_counts = init_plot["initial_status"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 4))
    status_counts.plot(kind="bar", ax=ax, color="tab:green")
    ax.set_title("Initial status distribution")
    ax.set_xlabel("initial status")
    ax.set_ylabel("number of generators")
    plt.tight_layout()
    plt.show()

resp_heat = resp_plot.copy()
hour_col = "hour" if "hour" in resp_heat.columns else resp_heat.columns[0]
resp_numeric = resp_heat.drop(columns=[hour_col], errors="ignore")
resp_numeric = resp_numeric.apply(pd.to_numeric, errors="coerce").fillna(0)

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(resp_numeric.T, aspect="auto", interpolation="nearest", cmap="Greys")
ax.set_title("Commitment heatmap for one instance")
ax.set_xlabel("hour")
ax.set_ylabel("generator")
plt.colorbar(im, ax=ax, label="on/off")
plt.tight_layout()
plt.show()

on_per_hour = resp_numeric.sum(axis=1)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(on_per_hour, lw=2, color="tab:red")
ax.set_title("Number of committed generators per hour")
ax.set_xlabel("hour")
ax.set_ylabel("units on")
plt.tight_layout()
plt.show()

In [ ]:
def prepare_eda_frames(exp_df, init_df, resp_df):
    exp_plot = exp_df.copy()
    init_plot = init_df.copy()
    resp_plot = resp_df.copy()

    if "Unnamed: 0" in exp_plot.columns:
        exp_plot = exp_plot.rename(columns={"Unnamed: 0": "hour"})
    if "Unnamed: 0" in init_plot.columns:
        init_plot = init_plot.rename(columns={"Unnamed: 0": "generator"})
    if "Unnamed: 0" in resp_plot.columns:
        resp_plot = resp_plot.rename(columns={"Unnamed: 0": "hour"})

    hour_col = "hour" if "hour" in resp_plot.columns else resp_plot.columns[0]
    resp_numeric = resp_plot.drop(columns=[hour_col], errors="ignore")
    resp_numeric = resp_numeric.apply(pd.to_numeric, errors="coerce").fillna(0)
    return exp_plot, init_plot, resp_plot, resp_numeric


def plot_instance_eda(instance_id, data_root=DATA_ROOT, save_path=None):
    sample = read_instance(data_root / instance_id)

    exp_key, exp_df = first_sheet(sample["explanatory"], keywords=("profile",))
    init_key, init_df = first_sheet(sample["explanatory"], keywords=("initial",))
    resp_key, resp_df = first_sheet(sample["response"])

    exp_plot, init_plot, resp_plot, resp_numeric = prepare_eda_frames(exp_df, init_df, resp_df)

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))

    ax = axes[0, 0]
    ax.plot(exp_plot["demand"], label="demand", lw=2)
    ax.plot(exp_plot["wind"], label="wind", lw=2)
    ax.plot(exp_plot["solar"], label="solar", lw=2)
    ax.plot(exp_plot["wind"] + exp_plot["solar"], label="wind + solar", lw=2, linestyle="--")
    ax.set_title("72-hour demand and renewables")
    ax.set_xlabel("hour")
    ax.set_ylabel("MW")
    ax.legend()
    ax.grid(True)

    ax = axes[0, 1]
    if "initial_status" in init_plot.columns:
        status_counts = init_plot["initial_status"].value_counts().sort_index()
        status_counts.plot(kind="bar", ax=ax, color="tab:green")
        ax.set_title("Initial status distribution")
        ax.set_xlabel("initial status")
        ax.set_ylabel("number of generators")
    else:
        ax.text(0.5, 0.5, "initial_status not found", ha="center", va="center")
    ax.grid(True)

    ax = axes[1, 0]
    im = ax.imshow(resp_numeric.T, aspect="auto", interpolation="nearest", cmap="Greys")
    ax.set_title("Commitment heatmap")
    ax.set_xlabel("hour")
    ax.set_ylabel("generator")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="on/off")

    ax = axes[1, 1]
    on_per_hour = resp_numeric.sum(axis=1)
    ax.plot(on_per_hour, lw=2, color="tab:red")
    ax.set_title("Committed generators per hour")
    ax.set_xlabel("hour")
    ax.set_ylabel("units on")
    ax.grid(True)

    fig.suptitle(f"EDA summary for {instance_id}", fontsize=16, y=1.02)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")

    plt.show()

chosen_instance = "instance_2022_Q1_1"
plot_instance_eda(chosen_instance)
chosen_instance = "instance_2022_Q1_7"
plot_instance_eda(chosen_instance)

In [ ]:
def extract_thermal_capacity_map(input_json):
    cap_map = {}
    gens = input_json.get("Generators", {}) if isinstance(input_json, dict) else {}

    for gen_name, gen_info in gens.items():
        if not isinstance(gen_info, dict):
            continue
        gtype = str(gen_info.get("Type", "")).strip().lower()
        if gtype != "thermal":
            continue

        cap = None
        pcurve = gen_info.get("Production cost curve (MW)")
        if isinstance(pcurve, list) and len(pcurve) > 0:
            try:
                cap = float(np.max(pd.to_numeric(pd.Series(pcurve), errors="coerce")))
            except Exception:
                cap = None

        if cap is None:
            for key in ["Maximum power (MW)", "Max Capacity", "Pmax", "Maximum capacity"]:
                if key in gen_info:
                    try:
                        val = gen_info[key]
                        if isinstance(val, list):
                            cap = float(np.max(pd.to_numeric(pd.Series(val), errors="coerce")))
                        else:
                            cap = float(val)
                        break
                    except Exception:
                        pass

        if cap is not None and np.isfinite(cap):
            cap_map[str(gen_name)] = float(cap)

    return cap_map


def plot_uc_dispatch_stack(instance_id, data_root=DATA_ROOT, commitment_override=None, save_path=None, max_generators=20):
    sample = read_instance(Path(data_root) / instance_id)

    exp_key, exp_df = first_sheet(sample["explanatory"], keywords=("profile",))
    init_key, init_df = first_sheet(sample["explanatory"], keywords=("initial",))
    resp_key, resp_df = first_sheet(sample["response"])

    exp_plot, init_plot, resp_plot, resp_numeric = prepare_eda_frames(exp_df, init_df, resp_df)

    demand = np.asarray(exp_plot["demand"], dtype=float)
    wind = np.asarray(exp_plot["wind"], dtype=float)
    solar = np.asarray(exp_plot["solar"], dtype=float)
    net_demand = np.clip(demand - wind - solar, 0.0, None)

    if commitment_override is None:
        commitment = resp_numeric.copy()
    else:
        commitment = pd.DataFrame(commitment_override).apply(pd.to_numeric, errors="coerce").fillna(0)

    cap_map = extract_thermal_capacity_map(sample["input_json"])

    gen_names = list(commitment.columns)
    cap_series = pd.Series([cap_map.get(str(g), np.nan) for g in gen_names], index=gen_names, dtype=float)

    print("Generators in response:", len(gen_names))
    print("Thermal capacities found:", int(cap_series.notna().sum()))
    print("Missing capacities:", int(cap_series.isna().sum()))

    has_capacity = cap_series.notna().sum() > 0

    fig, axes = plt.subplots(2, 1, figsize=(18, 11), sharex=True,
                             gridspec_kw={"height_ratios": [1, 1.25]})

    ax = axes[0]
    ax.plot(demand, label="demand", lw=2, color="tab:blue")
    ax.plot(net_demand, label="net demand", lw=2, color="tab:orange")
    ax.set_title(f"Demand vs net demand: {instance_id}")
    ax.set_ylabel("MW")
    ax.legend()
    ax.grid(True)

    ax = axes[1]

    if has_capacity:
        mw_by_gen = commitment.copy()
        for g in mw_by_gen.columns:
            cap = cap_series.get(g, np.nan)
            mw_by_gen[g] = mw_by_gen[g] * (cap if pd.notna(cap) else 0.0)

        total_contrib = mw_by_gen.sum(axis=0).sort_values(ascending=False)
        keep_n = min(max_generators, len(total_contrib))
        keep_gens = total_contrib.head(keep_n).index.tolist()
        other_cols = [g for g in mw_by_gen.columns if g not in keep_gens]

        mw_plot = mw_by_gen[keep_gens].copy()
        if len(other_cols) > 0:
            mw_plot["other"] = mw_by_gen[other_cols].sum(axis=1)

        hours = np.arange(len(mw_plot))
        bottom = np.zeros(len(mw_plot), dtype=float)
        colors = plt.cm.tab20(np.linspace(0, 1, max(1, mw_plot.shape[1])))

        display_names = {}
        for g in mw_plot.columns:
            if g == "other":
                display_names[g] = "other"
            else:
                cap = cap_series.get(g, np.nan)
                if pd.notna(cap):
                    display_names[g] = f"{g} ({cap:.0f} MW)"
                else:
                    display_names[g] = f"{g} (n/a)"

        for i, g in enumerate(mw_plot.columns):
            vals = mw_plot[g].to_numpy(dtype=float)
            ax.bar(
                hours,
                vals,
                bottom=bottom,
                width=0.9,
                label=display_names[g],
                color=colors[i % len(colors)]
            )
            bottom += vals

        ax.plot(net_demand, color="black", lw=2.0, linestyle="--", label="net demand")
        feasible_hours = bottom >= net_demand
        ax.text(
            0.01, 0.98,
            f"Hours meeting net demand: {int(feasible_hours.sum())}/{len(feasible_hours)}",
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=10,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none")
        )

        ax.set_title("Stacked committed capacity by generator")
        ax.set_xlabel("hour")
        ax.set_ylabel("MW")
        ax.grid(True, axis="y")
        ax.legend(ncol=2, fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")


    else:
        on_per_hour = commitment.sum(axis=1)
        ax.bar(np.arange(len(on_per_hour)), on_per_hour, color="tab:red")
        ax.set_title("Units on per hour (thermal capacities not found)")
        ax.set_xlabel("hour")
        ax.set_ylabel("units on")
        ax.grid(True, axis="y")

    

    fig.suptitle(f"UC dispatch stack for {instance_id}", fontsize=16, y=1.02)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")

    plt.show()


plot_uc_dispatch_stack("instance_2022_Q1_8")
plot_uc_dispatch_stack("instance_2022_Q1_7")

## 4B. Episode fingerprint for comparison across instances

In [ ]:
def plot_episode_fingerprint(instance_id: str, data_root: Path = DATA_ROOT, save_path=None):
    sample_local = read_instance(Path(data_root) / instance_id)
    exp_key, exp_df_local = first_sheet(sample_local["explanatory"], keywords=("profile",))
    init_key, init_df_local = first_sheet(sample_local["explanatory"], keywords=("initial",))
    resp_key, resp_df_local = first_sheet(sample_local["response"])

    exp_plot, init_plot, resp_plot, resp_numeric = prepare_eda_frames(exp_df_local, init_df_local, resp_df_local)
    net_demand = np.clip(exp_plot["demand"] - exp_plot["wind"] - exp_plot["solar"], 0.0, None)

    summary_rows = [
        ("episode id", instance_id),
        ("peak demand (MW)", f"{float(exp_plot['demand'].max()):,.1f}"),
        ("min demand (MW)", f"{float(exp_plot['demand'].min()):,.1f}"),
        ("peak net demand (MW)", f"{float(net_demand.max()):,.1f}"),
        ("mean demand (MW)", f"{float(exp_plot['demand'].mean()):,.1f}"),
        ("mean wind (MW)", f"{float(exp_plot['wind'].mean()):,.1f}"),
        ("mean solar (MW)", f"{float(exp_plot['solar'].mean()):,.1f}"),
        ("initial units on", str(int((init_plot["initial_status"] > 0).sum())) if "initial_status" in init_plot.columns else "n/a"),
        ("total commitment changes", f"{float(resp_numeric.diff().abs().fillna(0).sum().sum()):,.0f}" if len(resp_numeric) else "n/a"),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"width_ratios": [2.2, 1.0]})

    ax = axes[0]
    ax.plot(exp_plot["demand"], label="demand", lw=2)
    ax.plot(net_demand, label="net demand", lw=2)
    ax.set_title(f"Demand vs net demand: {instance_id}")
    ax.set_xlabel("hour")
    ax.set_ylabel("MW")
    ax.legend()
    ax.grid(True)

    ax = axes[1]
    ax.axis("off")
    table = ax.table(
        cellText=[[k, v] for k, v in summary_rows],
        colLabels=["metric", "value"],
        cellLoc="left",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.35)
    ax.set_title("Episode summary", pad=12)

    fig.suptitle(f"UC episode fingerprint for {instance_id}", fontsize=15, y=1.03)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()


chosen_instance = "instance_2022_Q1_7"
episode_summary_df = plot_episode_fingerprint(chosen_instance)
chosen_instance = "instance_2022_Q1_8"
episode_summary_df = plot_episode_fingerprint(chosen_instance)


# 4.C. Battery

In [ ]:
def _hourly_frame_from_json(obj, hour_name="hour"):
    if obj is None:
        return pd.DataFrame()

    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
    elif isinstance(obj, dict):
        df = pd.DataFrame(obj)
    else:
        return pd.DataFrame()

    df = df.copy()

    if hour_name not in df.columns:
        df.insert(0, hour_name, np.arange(1, len(df) + 1))

    df[hour_name] = pd.to_numeric(df[hour_name], errors="coerce")

    for c in df.columns:
        if c != hour_name:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    return df


def plot_storage_eda(instance_id, data_root=DATA_ROOT, save_path=None):
    sample_local = read_instance(Path(data_root) / instance_id)

    input_tables = input_json_to_dfs(sample_local["input_json"])
    storage_df = input_tables["storage_df"].copy()
    output_json = sample_local["output_json"]

    if storage_df.empty:
        print(f"No storage units found in {instance_id}")
        return

    if "name" not in storage_df.columns:
        if "Unnamed: 0" in storage_df.columns:
            storage_df = storage_df.rename(columns={"Unnamed: 0": "name"})
        else:
            storage_df.insert(0, "name", storage_df.index.astype(str))

    storage_df["name"] = storage_df["name"].astype(str)

    numeric_cols = [
        "Maximum level (MWh)",
        "Initial level (MWh)",
        "Last period minimum level (MWh)",
        "Maximum charge rate (MW)",
        "Maximum discharge rate (MW)",
        "Charge efficiency",
        "Discharge efficiency",
        "Charge cost ($/MW)",
        "Discharge cost ($/MW)",
    ]
    for col in numeric_cols:
        if col in storage_df.columns:
            storage_df[col] = pd.to_numeric(storage_df[col], errors="coerce")

    sort_col = "Maximum level (MWh)" if "Maximum level (MWh)" in storage_df.columns else storage_df.columns[0]
    storage_df = storage_df.sort_values(sort_col, ascending=False, na_position="last").reset_index(drop=True)

    summary_rows = [
        ("instance id", instance_id),
        ("storage units", int(len(storage_df))),
        ("total max storage (MWh)", f"{float(storage_df['Maximum level (MWh)'].fillna(0).sum()):,.1f}" if "Maximum level (MWh)" in storage_df.columns else "n/a"),
        ("mean initial level (MWh)", f"{float(storage_df['Initial level (MWh)'].fillna(0).mean()):,.1f}" if "Initial level (MWh)" in storage_df.columns else "n/a"),
        ("mean charge eff.", f"{float(storage_df['Charge efficiency'].dropna().mean()):.3f}" if "Charge efficiency" in storage_df.columns and storage_df["Charge efficiency"].notna().any() else "n/a"),
        ("mean discharge eff.", f"{float(storage_df['Discharge efficiency'].dropna().mean()):.3f}" if "Discharge efficiency" in storage_df.columns and storage_df["Discharge efficiency"].notna().any() else "n/a"),
        ("simultaneous charge/discharge allowed",
         int(storage_df["Allow simultaneous charging and discharging"].fillna(0).astype(int).sum())
         if "Allow simultaneous charging and discharging" in storage_df.columns else "n/a"),
    ]

    display(pd.DataFrame(summary_rows, columns=["metric", "value"]))

    level_df = _hourly_frame_from_json(output_json.get("Storage level (MWh)"))
    charge_df = _hourly_frame_from_json(output_json.get("Storage charging rates (MW)"))
    discharge_df = _hourly_frame_from_json(output_json.get("Storage discharging rates (MW)"))

    if level_df.empty:
        print("Storage level data not found in OutputData.json.gz")
        return

    level_units = [c for c in level_df.columns if c != "hour"]
    charge_units = [c for c in charge_df.columns if c != "hour"] if not charge_df.empty else []
    discharge_units = [c for c in discharge_df.columns if c != "hour"] if not discharge_df.empty else []

    charge_units = [c for c in charge_units if c in level_units]
    discharge_units = [c for c in discharge_units if c in level_units]

    level_df = level_df.sort_values("hour").reset_index(drop=True)
    if not charge_df.empty:
        charge_df = charge_df.sort_values("hour").reset_index(drop=True)
    if not discharge_df.empty:
        discharge_df = discharge_df.sort_values("hour").reset_index(drop=True)

    fig, axes = plt.subplots(2, 2, figsize=(20, 12))

    ax = axes[0, 0]
    y = np.arange(len(storage_df))
    max_level = storage_df["Maximum level (MWh)"] if "Maximum level (MWh)" in storage_df.columns else pd.Series(np.zeros(len(storage_df)))
    init_level = storage_df["Initial level (MWh)"] if "Initial level (MWh)" in storage_df.columns else pd.Series(np.zeros(len(storage_df)))

    ax.barh(y - 0.2, max_level.fillna(0), height=0.4, label="max level (MWh)", color="tab:blue")
    ax.barh(y + 0.2, init_level.fillna(0), height=0.4, label="initial level (MWh)", color="tab:orange")
    ax.set_yticks(y)
    ax.set_yticklabels(storage_df["name"])
    ax.set_xlabel("MWh")
    ax.set_title("Storage capacity and initial level by unit")
    ax.legend()
    ax.grid(True, axis="x", alpha=0.3)

    ax = axes[0, 1]
    max_charge = storage_df["Maximum charge rate (MW)"] if "Maximum charge rate (MW)" in storage_df.columns else pd.Series(np.zeros(len(storage_df)))
    max_discharge = storage_df["Maximum discharge rate (MW)"] if "Maximum discharge rate (MW)" in storage_df.columns else pd.Series(np.zeros(len(storage_df)))

    ax.barh(y - 0.2, max_charge.fillna(0), height=0.4, label="max charge rate (MW)", color="tab:green")
    ax.barh(y + 0.2, max_discharge.fillna(0), height=0.4, label="max discharge rate (MW)", color="tab:red")
    ax.set_yticks(y)
    ax.set_yticklabels(storage_df["name"])
    ax.set_xlabel("MW")
    ax.set_title("Storage charge / discharge limits by unit")
    ax.legend()
    ax.grid(True, axis="x", alpha=0.3)

    ax = axes[1, 0]
    level_matrix = level_df[level_units].to_numpy().T
    im = ax.imshow(level_matrix, aspect="auto", interpolation="nearest", cmap="Blues")
    ax.set_title("Storage level heatmap over 72 hours")
    ax.set_xlabel("hour")
    ax.set_ylabel("storage unit")
    ax.set_yticks(np.arange(len(level_units)))
    ax.set_yticklabels(level_units)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="level (MWh)")

    ax = axes[1, 1]
    total_level = level_df[level_units].sum(axis=1)
    ax.plot(level_df["hour"], total_level, lw=2, label="total storage level (MWh)", color="tab:blue")

    if not charge_df.empty and len(charge_units) > 0:
        total_charge = charge_df[charge_units].sum(axis=1)
        ax.plot(charge_df["hour"], total_charge, lw=2, label="total charging (MW)", color="tab:green")

    if not discharge_df.empty and len(discharge_units) > 0:
        total_discharge = discharge_df[discharge_units].sum(axis=1)
        ax.plot(discharge_df["hour"], total_discharge, lw=2, label="total discharging (MW)", color="tab:red")

    ax.set_title("Aggregate storage behaviour over time")
    ax.set_xlabel("hour")
    ax.set_ylabel("MW / MWh")
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.suptitle(f"Storage / battery EDA for {instance_id}", fontsize=16, y=1.02)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")

    plt.show()


chosen_instance = "instance_2022_Q1_8"
plot_storage_eda(chosen_instance)
chosen_instance = "instance_2022_Q1_7"
plot_storage_eda(chosen_instance)

In [ ]:
import re
from collections import defaultdict

def parse_instance_name(name: str):
    m = re.match(r"instance_(\d{4})_Q(\d)_(\d+)", name)
    if not m:
        return None
    return {
        "instance_id": name,
        "year": int(m.group(1)),
        "quarter": int(m.group(2)),
        "index": int(m.group(3)),
    }


def get_storage_summary_from_input_json(input_json):
    stor = input_json.get("Storage units", {}) if isinstance(input_json, dict) else {}
    rows = []

    for name, info in stor.items():
        if not isinstance(info, dict):
            continue

        row = {
            "name": name,
            "max_level_mwh": np.nan,
            "init_level_mwh": np.nan,
            "max_charge_mw": np.nan,
            "max_discharge_mw": np.nan,
            "charge_eff": np.nan,
            "discharge_eff": np.nan,
            "allow_simultaneous": np.nan,
        }

        for key, target in [
            ("Maximum level (MWh)", "max_level_mwh"),
            ("Initial level (MWh)", "init_level_mwh"),
            ("Maximum charge rate (MW)", "max_charge_mw"),
            ("Maximum discharge rate (MW)", "max_discharge_mw"),
            ("Charge efficiency", "charge_eff"),
            ("Discharge efficiency", "discharge_eff"),
            ("Allow simultaneous charging and discharging", "allow_simultaneous"),
        ]:
            if key in info:
                try:
                    row[target] = float(info[key])
                except Exception:
                    row[target] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


battery_rows = []

for inst_dir in instance_dirs:
    try:
        sample_inst = read_instance(inst_dir)
        meta = parse_instance_name(inst_dir.name)

        stor_df = get_storage_summary_from_input_json(sample_inst["input_json"])
        if stor_df.empty:
            battery_rows.append({
                "instance_id": inst_dir.name,
                "year": meta["year"] if meta else np.nan,
                "quarter": meta["quarter"] if meta else np.nan,
                "n_storage_units": 0,
                "total_storage_capacity_mwh": 0.0,
                "total_initial_level_mwh": 0.0,
                "mean_charge_eff": np.nan,
                "mean_discharge_eff": np.nan,
            })
            continue

        battery_rows.append({
            "instance_id": inst_dir.name,
            "year": meta["year"] if meta else np.nan,
            "quarter": meta["quarter"] if meta else np.nan,
            "n_storage_units": int(len(stor_df)),
            "total_storage_capacity_mwh": float(stor_df["max_level_mwh"].fillna(0).sum()),
            "total_initial_level_mwh": float(stor_df["init_level_mwh"].fillna(0).sum()),
            "mean_charge_eff": float(stor_df["charge_eff"].dropna().mean()) if stor_df["charge_eff"].notna().any() else np.nan,
            "mean_discharge_eff": float(stor_df["discharge_eff"].dropna().mean()) if stor_df["discharge_eff"].notna().any() else np.nan,
        })

    except Exception as e:
        battery_rows.append({
            "instance_id": inst_dir.name,
            "year": np.nan,
            "quarter": np.nan,
            "n_storage_units": np.nan,
            "total_storage_capacity_mwh": np.nan,
            "total_initial_level_mwh": np.nan,
            "mean_charge_eff": np.nan,
            "mean_discharge_eff": np.nan,
            "error": str(e),
        })

battery_df = pd.DataFrame(battery_rows)

display(battery_df.head())
print("Total instances scanned:", len(battery_df))
print("Instances with storage:", int((battery_df["n_storage_units"].fillna(0) > 0).sum()))


fig, axes = plt.subplots(2, 2, figsize=(18, 10))

ax = axes[0, 0]
ax.hist(battery_df["n_storage_units"].dropna(), bins=20, color="tab:blue", edgecolor="white")
ax.set_title("Distribution of number of storage units per instance")
ax.set_xlabel("number of storage units")
ax.set_ylabel("count")
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.hist(battery_df["total_storage_capacity_mwh"].dropna(), bins=20, color="tab:orange", edgecolor="white")
ax.set_title("Distribution of total storage capacity per instance")
ax.set_xlabel("total storage capacity (MWh)")
ax.set_ylabel("count")
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.scatter(
    battery_df["total_storage_capacity_mwh"],
    battery_df["total_initial_level_mwh"],
    s=18,
    alpha=0.7,
    color="tab:green",
)
ax.set_title("Initial storage level vs total storage capacity")
ax.set_xlabel("total storage capacity (MWh)")
ax.set_ylabel("total initial level (MWh)")
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
year_quarter_cap = (
    battery_df.dropna(subset=["year", "quarter"])
    .groupby(["year", "quarter"])["total_storage_capacity_mwh"]
    .mean()
    .reset_index()
)
if len(year_quarter_cap):
    year_quarter_pivot = year_quarter_cap.pivot(index="year", columns="quarter", values="total_storage_capacity_mwh")
    sns.heatmap(
        year_quarter_pivot,
        annot=True,
        fmt=".0f",
        cmap="YlOrBr",
        ax=ax,
        cbar_kws={"label": "mean storage capacity (MWh)"},
    )
    ax.set_title("Mean storage capacity by year and quarter")
    ax.set_xlabel("quarter")
    ax.set_ylabel("year")
else:
    ax.text(0.5, 0.5, "No year-quarter data", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Convert one instance into an RL-ready episode record

In [ ]:
def extract_numeric_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def as_long_table(df: pd.DataFrame, instance_id: str, sheet_name: str) -> pd.DataFrame:
    tmp = df.copy()
    tmp.insert(0, "instance_id", instance_id)
    tmp.insert(1, "sheet_name", sheet_name)
    return tmp


def response_to_matrix(response_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    key, df = first_sheet(response_dict)
    df = df.copy()

    if df.shape[1] > 1:
        first_col = df.columns[0]
        if str(first_col).lower() not in {"hour", "period"}:
            df = df.set_index(first_col)

    return df


def episode_summary(inst: dict) -> dict:
    exp = inst["explanatory"]
    resp = inst["response"]
    inp = inst["input_json"]
    out = inst["output_json"]

    profile_key, profile_df = first_sheet(exp, keywords=("profile",))
    init_key, init_df = first_sheet(exp, keywords=("initial",))
    resp_key, resp_df = first_sheet(resp)

    response_matrix = response_to_matrix(resp)

    return {
        "instance_id": inst["instance_id"],
        "profile_sheet": profile_key,
        "initial_sheet": init_key,
        "response_sheet": resp_key,
        "n_profile_rows": int(profile_df.shape[0]),
        "n_profile_cols": int(profile_df.shape[1]),
        "n_initial_rows": int(init_df.shape[0]),
        "n_initial_cols": int(init_df.shape[1]),
        "n_response_rows": int(resp_df.shape[0]),
        "n_response_cols": int(resp_df.shape[1]),
        "response_shape_clean": tuple(response_matrix.shape),
        "input_json_keys": len(inp) if hasattr(inp, "__len__") else None,
        "output_json_keys": len(out) if hasattr(out, "__len__") else None,
    }


summary = episode_summary(sample)
pd.Series(summary)

## 6. Build a full RL-ready dataset

In [ ]:
def split_instances(paths, train_frac=0.7, val_frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    paths = np.array(list(paths), dtype=object)
    rng.shuffle(paths)
    n = len(paths)
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))
    train = paths[:n_train].tolist()
    val = paths[n_train:n_train + n_val].tolist()
    test = paths[n_train + n_val:].tolist()
    return {"train": train, "val": val, "test": test}


def table_to_episode_features(exp_dict: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    out = {}
    for sheet, df in exp_dict.items():
        out[standardize_sheet_name(sheet)] = extract_numeric_table(df)
    return out


def instance_to_bundle(folder: Path) -> dict:
    inst = read_instance(folder)
    exp = table_to_episode_features(inst["explanatory"])
    resp = {standardize_sheet_name(k): v.copy() for k, v in inst["response"].items()}

    profile_key, profile_df = first_sheet(inst["explanatory"], keywords=("profile",))
    initial_key, initial_df = first_sheet(inst["explanatory"], keywords=("initial",))
    response_key, response_df = first_sheet(inst["response"])

    bundle = {
        "instance_id": inst["instance_id"],
        "folder": str(folder),
        "explanatory": exp,
        "response": resp,
        "input_json": inst["input_json"],
        "output_json": inst["output_json"],
        "profile_sheet_name": profile_key,
        "initial_sheet_name": initial_key,
        "response_sheet_name": response_key,
        "profile_table": profile_df,
        "initial_table": initial_df,
        "response_table": response_df,
    }
    return bundle


split_map = split_instances(instance_dirs, TRAIN_FRAC, VAL_FRAC, RNG_SEED)
for split, items in split_map.items():
    print(split, len(items))

print("Example train instance:", Path(split_map["train"][0]).name)

In [ ]:
def save_bundle(bundle: dict, split: str, index: int):
    split_dir = OUT_ROOT / split
    split_dir.mkdir(parents=True, exist_ok=True)

    inst_dir = split_dir / bundle["instance_id"]
    inst_dir.mkdir(parents=True, exist_ok=True)

    bundle["profile_table"].to_csv(inst_dir / "profiles.csv", index=False)
    bundle["initial_table"].to_csv(inst_dir / "initial_conditions.csv", index=False)
    bundle["response_table"].to_csv(inst_dir / "response_variables.csv", index=False)

    with open(inst_dir / "InputData.json", "w", encoding="utf-8") as f:
        json.dump(bundle["input_json"], f, indent=2)
    with open(inst_dir / "OutputData.json", "w", encoding="utf-8") as f:
        json.dump(bundle["output_json"], f, indent=2)

    pickle_path = inst_dir / "episode_bundle.pkl"
    with open(pickle_path, "wb") as f:
        pickle.dump(bundle, f)

    return {
        "split": split,
        "instance_id": bundle["instance_id"],
        "folder": bundle["folder"],
        "bundle_path": str(pickle_path),
        "profile_rows": int(bundle["profile_table"].shape[0]),
        "initial_rows": int(bundle["initial_table"].shape[0]),
        "response_rows": int(bundle["response_table"].shape[0]),
    }


manifest_rows = []
for split, folders in split_map.items():
    for i, folder in enumerate(folders):
        bundle = instance_to_bundle(Path(folder))
        manifest_rows.append(save_bundle(bundle, split, i))

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUT_ROOT / "manifest.csv", index=False)
display(manifest.head())
print("Saved manifest to:", OUT_ROOT / "manifest.csv")

## 7. Make a simple loader for RL training

In [ ]:
class EpisodeDataset:
    def __init__(self, root: Path, split: str):
        self.root = Path(root)
        self.split = split
        self.manifest = pd.read_csv(self.root / "manifest.csv")
        self.manifest = self.manifest[self.manifest["split"] == split].reset_index(drop=True)

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        bundle_path = Path(row["bundle_path"])
        with open(bundle_path, "rb") as f:
            bundle = pickle.load(f)
        return bundle

    def sample(self, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        idx = int(rng.integers(0, len(self)))
        return self[idx]


train_ds = EpisodeDataset(OUT_ROOT, "train")
val_ds = EpisodeDataset(OUT_ROOT, "val")
test_ds = EpisodeDataset(OUT_ROOT, "test")

print("train episodes:", len(train_ds))
print("val episodes:", len(val_ds))
print("test episodes:", len(test_ds))

example = train_ds[0]
print("Example episode_id:", example["instance_id"])
print("Profile table shape:", example["profile_table"].shape)
print("Initial table shape:", example["initial_table"].shape)
print("Response table shape:", example["response_table"].shape)

## 8. What to feed into RL next

In [ ]:
print(manifest.groupby("split").size())
print()
print("Manifest columns:", list(manifest.columns))
print()
print("One example bundle keys:", list(example.keys()))